In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 設定圖表風格
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] # 讓圖表可以顯示中文
plt.rcParams['axes.unicode_minus'] = False

In [3]:
# 1. 讀取與處理資料
file_path = 'C:\\Users\\170134\\Desktop\\新增資料夾\\cleaned_data_for_analysis.csv'
print(f"正在讀取資料: {file_path} ...")
df = pd.read_csv(file_path)

# 確保是數值與日期格式
df['Year'] = pd.to_numeric(df['Year'])
df['Return'] = pd.to_numeric(df['Return'], errors='coerce') / 100 # 轉成小數點 (例如 5% -> 0.05)
df['S_Score'] = pd.to_numeric(df['S_Score'], errors='coerce')

print(f"回測樣本數: {len(df)} 筆")

正在讀取資料: C:\Users\170134\Desktop\新增資料夾\cleaned_data_for_analysis.csv ...
回測樣本數: 8688 筆


In [4]:
# 2. 建立 Lag 資料 (用去年的分數選今年的股)
# 先排序
df = df.sort_values(by=['Company', 'Year'])
# 產生 S_Score_Lag1
df['S_Score_Lag1'] = df.groupby('Company')['S_Score'].shift(1)
# 去除空值 (第一年沒資料不能買)
df_backtest = df.dropna(subset=['S_Score_Lag1', 'Return'])

print(f"回測樣本數: {len(df_backtest)} 筆")

回測樣本數: 6379 筆


In [9]:
# 3. 年度回測引擎 (Yearly Rebalancing)
years = df_backtest['Year'].unique()
years.sort()

strategy_returns = []
market_returns = []
valid_years = []

print("\n--- 年度績效明細 ---")
print(f"{'年度':<6} {'幸福企業(High S)':<12} {'市場平均':<10} {'勝負'}")
print("-" * 45)

for year in years:
    # 撈出該年度所有公司的資料
    yearly_data = df_backtest[df_backtest['Year'] == year].copy()
    
    # 如果該年公司數太少，跳過 (避免誤差)
    if len(yearly_data) < 50:
        continue
        
    # A. 設定選股門檻：S 分數前 20% (Top 20%)
    threshold = yearly_data['S_Score_Lag1'].quantile(0.80)
    
    # B. 挑出幸福企業 (High S Portfolio)
    portfolio = yearly_data[yearly_data['S_Score_Lag1'] >= threshold]

    # C. 計算報酬率 (假設等權重買進)
    # 策略報酬
    strat_ret = portfolio['Return'].mean()
    # 市場報酬 (Benchmark)
    mkt_ret = yearly_data['Return'].mean()

    strategy_returns.append(strat_ret)
    market_returns.append(mkt_ret)
    valid_years.append(year)

    # 顯示年度結果
    result = "Win 🏆" if strat_ret > mkt_ret else "Loss"
    print(f"{year:<6} {strat_ret*100:>6.2f}%      {mkt_ret*100:>6.2f}%     {result}")


--- 年度績效明細 ---
年度     幸福企業(High S) 市場平均       勝負
---------------------------------------------
2023    39.00%       35.03%     Win 🏆
2024    11.58%       15.01%     Loss
2025     9.74%        4.98%     Win 🏆


In [10]:
# 4. 計算累積報酬 (Cumulative Return)
# 假設初始資金 100 元
strategy_cum = (1 + pd.Series(strategy_returns)).cumprod() * 100
market_cum = (1 + pd.Series(market_returns)).cumprod() * 100

# 為了畫圖好看，加入起點 (100)
plot_years = [valid_years[0]-1] + valid_years
plot_strat = [100] + strategy_cum.tolist()
plot_mkt = [100] + market_cum.tolist()

In [11]:
# 5. 總結績效指標
total_ret_strat = plot_strat[-1] - 100
total_ret_mkt = plot_mkt[-1] - 100
win_rate = sum([1 for s, m in zip(strategy_returns, market_returns) if s > m]) / len(valid_years)

print("-" * 45)
print(f"累積總報酬 (策略): {total_ret_strat:.2f}%")
print(f"累積總報酬 (市場): {total_ret_mkt:.2f}%")
print(f"超額報酬 (Alpha): {total_ret_strat - total_ret_mkt:.2f}%")
print(f"勝率 (Win Rate): {win_rate*100:.0f}%")

---------------------------------------------
累積總報酬 (策略): 70.19%
累積總報酬 (市場): 63.03%
超額報酬 (Alpha): 7.16%
勝率 (Win Rate): 67%


In [ ]:
# 6. 畫圖 (Wealth Curve)
plt.figure(figsize=(10, 6))
plt.plot(plot_years, plot_strat, marker='o', color='#e74c3c', linewidth=3, label='幸福企業策略 (Top 20% S-Score)')
plt.plot(plot_years, plot_mkt, marker='s', color='#95a5a6', linewidth=2, linestyle='--', label='市場平均 (Benchmark)')

plt.fill_between(plot_years, plot_strat, plot_mkt, where=(pd.Series(plot_strat) > pd.Series(plot_mkt)), 
                 interpolate=True, color='red', alpha=0.1, label='超額獲利區')

plt.title('High-S Score Strategy Backtest (幸福企業 vs 大盤)', fontsize=15)
plt.xlabel('年份')
plt.ylabel('資產淨值 (起始=100)')
plt.legend()
plt.grid(True, alpha=0.3)
# plt.savefig('high_s_strategy.png')
plt.show()